In [21]:
myPortfolio = [186,187,188, 15077]

# port186 = Portfolio(186)
# # port186.get_all_days()
# dt186 = port186.df

# port187 = Portfolio(187)
# # port187.get_all_days()
# dt187 = port187.df

# port188 = Portfolio(188)
# # port188.get_all_days()
# dt188 = port188.df



In [22]:
import requests
from requests.structures import CaseInsensitiveDict
import numpy_financial as npf
import numpy as np
from datetime import datetime as dt
import pandas as pd

headers = CaseInsensitiveDict()
headers["accept"] = "application/json"

In [23]:
pd.options.display.float_format = '{:.0f}'.format

In [24]:
# Objeto portafolio, donde tendrá los datos base del mismo y los datos financieros principales 
# De momento no está en uso

class Portfolio:
    __baseUrlApi = "https://fintual.cl/api/real_assets/"

    def __init__(self, id):
        urlAssetInfo         = f"{self.__baseUrlApi}{id}"
        assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']
        
        self.id          = id
        self.name        = assetInfo['name']
        self.startDate   = assetInfo['start_date']
        self.lastDate    = assetInfo['last_day']['date']
        self.lastPrice   = assetInfo['last_day']["net_asset_value"]
        # self.df          = pd.DataFrame() 
        self.df          = self.get_all_days()
    

In [25]:
__baseUrlApi = "https://fintual.cl/api/real_assets/"
id = 186
urlAssetInfo         = f"{__baseUrlApi}{id}"
# Obtener los datos base del portafolio
assetInfo            = requests.get(urlAssetInfo,  headers=headers).json()['data']['attributes']
assetInfo

{'name': 'Risky Norris',
 'symbol': 'FM-FIN-RCN-A',
 'serie': 'A',
 'start_date': '2018-02-13',
 'end_date': None,
 'previous_asset_id': None,
 'last_day': {'net_asset_value': 3083.2207, 'date': '2024-12-25'},
 'conceptual_asset_id': 36}

In [26]:
# Url de la api de funtual necesario para obtener todos los valores existentes en el tiempo
urlAssetInfoDays  = f"{__baseUrlApi}{id}/days" #?to_date=2024-08-28

# obtenemos los datos de la api y lo pasamos a un DataFrame
assetInfoDays        = requests.get(urlAssetInfoDays,  headers=headers).json()['data']
df = pd.DataFrame(assetInfoDays)

# Nomalizamos los datos a Json
df = pd.json_normalize(df['attributes'])[['date', 'price', 'shareholders', 'total_assets', 'total_net_assets', 'outstanding_shares']]
# En caso que los datos del primer día aún no estén disponibles dejamos dicha fila fuera del DataFrame
# if pd.Series(np.isnan(df.tail(1)['total_assets']) == True).all():
#     df = df[1:-1]

# Elimina las filas las cuales no dispongan del datos clave 'total_assets'
df.dropna(subset=['total_assets'], inplace=True)

# Se cambia el formato de la columna fecha para generar filtros y crear nuevas columnas con los años, mese y día
df['date']  = pd.to_datetime(df['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day


In [27]:
df

,date,price,shareholders,total_assets,total_net_assets,outstanding_shares,year,month,day
2,2024-12-23,3060,56752,405994225456,296523563553,96902289,2024,12,23
3,2024-12-22,3030,56670,405604198416,292788493707,96632417,2024,12,22
4,2024-12-21,3030,56670,405603954525,292797835154,96632417,2024,12,21
5,2024-12-20,3030,56670,405603710633,292807176903,96632417,2024,12,20
6,2024-12-19,3014,56628,397444555975,290653605257,96449031,2024,12,19
...,...,...,...,...,...,...,...,...,...
2503,2018-02-17,1016,2,1133361,1133247,1115,2018,2,17
2504,2018-02-16,1016,2,1133340,1133263,1115,2018,2,16
2505,2018-02-15,1016,2,1133173,1133133,1115,2018,2,15
2506,2018-02-14,1013,2,1291954,985305,972,2018,2,14


In [28]:

# if pd.Series(np.isnan(df.tail(1)['total_assets']) == True).all():
#     df = df[1:-1]
    
# last_date = df.tail(1)['date']

# Cambio de nombre de las columas 
df.columns = ('fecha','precio','accionistas','activos_totales','activos_neto_totales','acciones_en_circulación', 'año', 'mes', 'día')

In [29]:
df = df[::-1].reset_index(drop=True)
df

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
0,2018-02-13,1004,1,708357,401533,400,2018,2,13
1,2018-02-14,1013,2,1291954,985305,972,2018,2,14
2,2018-02-15,1016,2,1133173,1133133,1115,2018,2,15
3,2018-02-16,1016,2,1133340,1133263,1115,2018,2,16
4,2018-02-17,1016,2,1133361,1133247,1115,2018,2,17
...,...,...,...,...,...,...,...,...,...
2501,2024-12-19,3014,56628,397444555975,290653605257,96449031,2024,12,19
2502,2024-12-20,3030,56670,405603710633,292807176903,96632417,2024,12,20
2503,2024-12-21,3030,56670,405603954525,292797835154,96632417,2024,12,21
2504,2024-12-22,3030,56670,405604198416,292788493707,96632417,2024,12,22


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2506 entries, 0 to 2505
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   fecha                    2506 non-null   datetime64[ns]
 1   precio                   2506 non-null   float64       
 2   accionistas              2506 non-null   float64       
 3   activos_totales          2506 non-null   float64       
 4   activos_neto_totales     2506 non-null   float64       
 5   acciones_en_circulación  2506 non-null   float64       
 6   año                      2506 non-null   int32         
 7   mes                      2506 non-null   int32         
 8   día                      2506 non-null   int32         
dtypes: datetime64[ns](1), float64(5), int32(3)
memory usage: 147.0 KB


In [31]:
# Obtener los valores entre dos fechas
mask = df[(df['fecha'] > '2021-02-16') & (df['fecha'] <= '2022-03-26')]
mask

,fecha,precio,accionistas,activos_totales,activos_neto_totales,acciones_en_circulación,año,mes,día
1100,2021-02-17,1961,27432,115500425689,95466355248,48677338,2021,2,17
1101,2021-02-18,1919,27491,111169150640,93729253875,48847077,2021,2,18
1102,2021-02-19,1919,27602,111646351947,93950379891,48968560,2021,2,19
1103,2021-02-20,1919,27602,111646503978,93947445993,48968560,2021,2,20
1104,2021-02-21,1918,27602,111646503978,93944383049,48968560,2021,2,21
...,...,...,...,...,...,...,...,...,...
1498,2022-03-22,1964,62431,273619208402,225771317017,114951152,2022,3,22
1499,2022-03-23,1936,62458,272004269089,222836563443,115092403,2022,3,23
1500,2022-03-24,1965,62470,275914582088,226110675868,115084196,2022,3,24
1501,2022-03-25,1953,62476,268701750344,224550123214,114989003,2022,3,25


In [63]:
# Dado dos fechas obtener el irr a partir del precio de dicho día
sec = df[(df['fecha']>= '2023-12-23') & (df['fecha']<= '2024-12-22')]
first = sec[sec['fecha'] == '2023-12-23']
last  = sec[sec['fecha'] == '2024-12-22']
f1 = first.iloc[-1]['precio']*-1
l1 = last.iloc[-1]['precio']

# first = df[df['fecha'] == '2023-12-23']
# last  = df[df['fecha'] == '2024-12-22']
# f1 = first.iloc[-1]['precio']*-1
# l1 = last.iloc[-1]['precio']

print(f1)
print(l1)
print()
print(sec.describe()['precio'])
print()
npf.irr([f1,l1])*100

-2264.1811
3029.92

count    366
mean    2648
min     2211
25%     2537
50%     2641
75%     2738
max     3122
std      191
Name: precio, dtype: float64



33.819684299988204

In [33]:
def get_first_last_price_from_df(info):
    fi = info.iloc[0 ]['precio']
    la = info.iloc[-1]['precio']
    return fi, la


def tir_two_dates(price1, price2, to_percent = True, rou = 2):
    tir = npf.irr([price1*-1, price2])
    if to_percent == True:
        tir = tir * 100
    tir = round(tir, rou)
    return tir

In [60]:
d1 = df.iloc[0 ]['año']
d2 = df.iloc[-1]['año']

anual_info = dict()

for year in range(d1, d2+1):
    di = dict()
    info = df[df['año'] == year]

    pi , pf = get_first_last_price_from_df(info)
    di['tir'] = tir_two_dates(pi,pf)

    for x , i in info['precio'].describe().items():
        di[x] = i

    m1 = info.iloc[0 ]['mes']
    m2 = info.iloc[-1]['mes']
    
    month_info = dict()
    for month in range(m1, m2+1):
        mo = dict()
        minfo = info[info['mes'] == month]
        # print(minfo)

        mi , mf = get_first_last_price_from_df(minfo)
        mo['tir'] = tir_two_dates(mi,mf)

        # for y , j in minfo['precio'].describe().items():
        #     mo[y] = j
        # month_info[month] = mo

        month_info[month] = minfo['precio'].describe()
    
    di['months'] = month_info

    anual_info[year] = di

# anual_info[2018]['months'][2]['25%']
anual_info


{2018: {'tir': 4.15,
  'count': 322.0,
  'mean': 1091.6362804347825,
  'std': 65.48643250679008,
  'min': 973.6904,
  '25%': 1035.56225,
  '50%': 1097.50065,
  '75%': 1138.33695,
  'max': 1226.69,
  'months': {2: count     16
   mean    1013
   std        5
   min     1004
   25%     1009
   50%     1015
   75%     1016
   max     1024
   Name: precio, dtype: float64,
   3: count     31
   mean    1019
   std       22
   min      983
   25%      999
   50%     1022
   75%     1038
   max     1049
   Name: precio, dtype: float64,
   4: count     30
   mean     993
   std        8
   min      974
   25%      990
   50%      993
   75%     1000
   max     1006
   Name: precio, dtype: float64,
   5: count     31
   mean    1052
   std       18
   min     1006
   25%     1047
   50%     1058
   75%     1064
   max     1073
   Name: precio, dtype: float64,
   6: count     30
   mean    1092
   std        9
   min     1075
   25%     1086
   50%     1094
   75%     1099
   max     1104
   Nam

# Siguientes objetivos
o - Irr en el año
o - Irr entre dos fechas
o - Irr meses en el año
- Formatear números
- Crear validaciones 
- Crear clases y funciones correspondientes
- Escritura y lectura de excel con los datos, para no recurrir %100 de la API y dicho archivo poderlo usar en Excel y Power Bi
- Generación de diagramas en fechas especificas y con input
- Simular ingreseo y egreso de dinero al invertir en cierta fecha
- Forma de buscar mayor beneficio si se invirtío en cierta fecha
- Generar reportes completos

